# BÀI TOÁN PHÂN LOẠI NHỊ PHÂN TRÊN DỮ LIỆU THỜI TIẾT (WEATHER OPEN-METEO)
## Triển khai Logistic Regression từ đầu bằng NumPy & So sánh với Scikit-Learn

### Mục tiêu của Notebook:
1. **Khám phá và Tiền xử lý dữ liệu (EDA & Data Preprocessing)** trên tập dữ liệu thời tiết thực tế `weather_openmeteo.csv`.
2. **Kỹ thuật trích xuất đặc trưng (Feature Engineering)**: Bóc tách giờ trong ngày, tháng trong năm, xử lý chuẩn hóa đặc trưng.
3. **Định nghĩa bài toán phân loại nhị phân**:
   - **Lớp 0 (Không mưa / No Rain)**: Trời quang, nhiều mây, âm u (`Clear sky`, `Mainly clear`, `Partly cloudy`, `Overcast`).
   - **Lớp 1 (Có mưa / Rain)**: Mưa phùn, mưa rào, mưa lớn (`Light drizzle`, `Moderate drizzle`, `Dense drizzle`, `Slight rain`, `Moderate rain`, `Heavy rain`).
4. **Cài đặt mô hình Logistic Regression thuần NumPy (Scratch)**:
   - Hàm kích hoạt Sigmoid có chống tràn số (numerical stability).
   - Hàm mất mát Binary Cross-Entropy Loss.
   - Tối ưu hóa bằng thuật toán Gradient Descent theo công thức đạo hàm đã chứng minh trong file `preliminaries.ipynb`: $\nabla_w L = \frac{1}{N} X^T (\hat{y} - y)$.
5. **Trực quan hóa đường cong hàm mất mát (Loss Curve)** qua các vòng lặp huấn luyện.
6. **Đánh giá mô hình toàn diện**: Tính toán ma trận nhầm lẫn (Confusion Matrix) và các độ đo **Accuracy, Precision, Recall, F1-Score**.
7. **Thực nghiệm đối sánh với thư viện Scikit-Learn** (theo tinh thần của Assignment 3) và phân tích ý nghĩa trọng số của các đặc trưng thời tiết.

In [ ]:
# Import các thư viện cần thiết
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Cấu hình đồ họa hiển thị
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['font.size'] = 11

## Bước 1: Tải và Khám phá Dữ liệu (Exploratory Data Analysis - EDA)

In [ ]:
# Đọc dữ liệu từ file weather_openmeteo.csv
df = pd.read_csv('weather_openmeteo.csv')
print(f"Tổng số dòng và cột: {df.shape[0]:,} dòng, {df.shape[1]} cột")
print("\n5 dòng dữ liệu đầu tiên:")
display(df.head())

print("\nKiểm tra kiểu dữ liệu và giá trị thiếu (Null values):")
print(df.info())

print("\nThống kê các trạng thái thời tiết trong cột 'Weather':")
weather_counts = df['Weather'].value_counts()
display(weather_counts)

## Bước 2: Tiền xử lý & Trích xuất đặc trưng (Feature Engineering)

1. **Trích xuất thời gian**:
   - `Hour`: Giờ trong ngày (0 đến 23) từ cột `Time`.
   - `Month`: Tháng trong năm (1 đến 12) từ cột `Date`.
2. **Tạo nhãn mục tiêu nhị phân (Target)**:
   - **1 (Có mưa)**: Các trạng thái có mưa (`drizzle`, `rain`).
   - **0 (Không mưa)**: Các trạng thái không mưa còn lại.
3. **Lựa chọn các đặc trưng đầu vào ($X$)**:
   - `Temperature`: Nhiệt độ (°C)
   - `Humidity`: Độ ẩm tương đối (%)
   - `Wind`: Tốc độ gió (km/h)
   - `Barometer`: Áp suất khí quyển (hPa)
   - `Hour`: Giờ trong ngày
   - `Month`: Tháng trong năm

In [ ]:
# 1. Trích xuất Hour và Month
df['Hour'] = df['Time'].apply(lambda t: int(str(t).split(':')[0]))
df['Date_dt'] = pd.to_datetime(df['Date'], errors='coerce')
df['Month'] = df['Date_dt'].dt.month

# 2. Gán nhãn nhị phân cho bài toán dự báo mưa
rain_types = [
    'Light drizzle', 'Moderate drizzle', 'Dense drizzle',
    'Slight rain', 'Moderate rain', 'Heavy rain'
]
df['Target'] = df['Weather'].isin(rain_types).astype(int)

# Thống kê nhãn nhị phân
label_summary = pd.DataFrame({
    'Số lượng': df['Target'].value_counts(),
    'Tỷ lệ (%)': df['Target'].value_counts(normalize=True) * 100
}, index=[0, 1])
label_summary.index = ['0 (Không mưa)', '1 (Có mưa)']
print("Phân phối nhãn nhị phân (Target):")
display(label_summary)

# Thống kê mô tả các đặc trưng đầu vào
feature_cols = ['Temperature', 'Humidity', 'Wind', 'Barometer', 'Hour', 'Month']
print("\nThống kê mô tả các đặc trưng số:")
display(df[feature_cols].describe())

## Bước 3: Phân chia tập huấn luyện/kiểm tra & Chuẩn hóa dữ liệu (Feature Scaling)

Vì các đặc trưng có thang đo rất khác nhau (ví dụ: `Barometer` ~ 1010, `Humidity` ~ 80, `Wind` ~ 5), Gradient Descent sẽ gặp khó khăn hoặc mất rất nhiều thời gian hội tụ nếu không chuẩn hóa.

Chúng ta áp dụng **Z-score Standardization** thuần bằng NumPy:
$$z = \frac{x - \mu}{\sigma}$$
*Lưu ý quan trọng*: $\mu$ và $\sigma$ chỉ được tính trên tập Train, sau đó dùng để biến đổi cả Train và Test nhằm tránh **Data Leakage**.

In [ ]:
# Lấy ma trận X và vector nhãn y
X = df[feature_cols].values
y = df['Target'].values

# Phân chia 80% Train, 20% Test (sử dụng NumPy hoán vị ngẫu nhiên)
np.random.seed(42)
indices = np.random.permutation(len(X))
train_size = int(0.8 * len(X))

train_idx = indices[:train_size]
test_idx = indices[train_size:]

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

# Chuẩn hóa dữ liệu bằng Z-score với NumPy
mean_train = np.mean(X_train, axis=0)
std_train = np.std(X_train, axis=0)
# Đảm bảo không chia cho 0
std_train = np.where(std_train == 0, 1e-8, std_train)

X_train_scaled = (X_train - mean_train) / std_train
X_test_scaled = (X_test - mean_train) / std_train

# Thêm hệ số chặn (Bias / Intercept term x0 = 1) vào ma trận đặc trưng
X_train_b = np.c_[np.ones(len(X_train_scaled)), X_train_scaled]
X_test_b = np.c_[np.ones(len(X_test_scaled)), X_test_scaled]

print(f"Tập huấn luyện (Train): {X_train_b.shape[0]} mẫu, {X_train_b.shape[1]} đặc trưng (đã gồm bias)")
print(f"Tập kiểm tra (Test):      {X_test_b.shape[0]} mẫu, {X_test_b.shape[1]} đặc trưng (đã gồm bias)")

## Bước 4: Xây dựng mô hình Logistic Regression từ đầu bằng NumPy (Scratch)

Dựa trên các kết quả dẫn xuất từ file `preliminaries.ipynb`:

1. **Mô hình tuyến tính và hàm Sigmoid**:
   $$z = Xw = w_0 + w_1 x_1 + ... + w_d x_d$$
   $$\hat{y} = \sigma(z) = \frac{1}{1 + e^{-z}}$$
   *Chống tràn số*: Giới hạn giá trị $z$ trong đoạn $[-500, 500]$ để tránh lỗi `Overflow` trong hàm `np.exp`.

2. **Hàm mất mát Binary Cross-Entropy Loss**:
   $$L(w) = -\frac{1}{N} \sum_{i=1}^N \left[ y^{(i)} \ln(\hat{y}^{(i)}) + (1 - y^{(i)}) \ln(1 - \hat{y}^{(i)}) \right]$$

3. **Đạo hàm (Gradient)**:
   $$\nabla_w L = \frac{\partial L}{\partial w} = \frac{1}{N} X^T (\hat{y} - y)$$

4. **Cập nhật trọng số với Gradient Descent**:
   $$w := w - \alpha \cdot \nabla_w L$$

In [ ]:
class LogisticRegressionNumPy:
    """
    Mô hình Logistic Regression được xây dựng từ đầu thuần NumPy.
    Tối ưu hóa bằng Gradient Descent.
    """
    def __init__(self, learning_rate=0.1, n_iterations=1000, verbose=True):
        self.learning_rate = learning_rate
        self.n_iterations = n_iterations
        self.verbose = verbose
        self.weights = None
        self.loss_history = []

    def _sigmoid(self, z):
        # Tránh tràn số bằng cách clip giá trị z
        z_clipped = np.clip(z, -500, 500)
        return 1.0 / (1.0 + np.exp(-z_clipped))

    def _compute_loss(self, y_true, y_pred, eps=1e-15):
        # Tránh log(0) bằng cách giới hạn y_pred trong khoảng [eps, 1-eps]
        y_pred_clipped = np.clip(y_pred, eps, 1.0 - eps)
        loss = -np.mean(y_true * np.log(y_pred_clipped) + (1.0 - y_true) * np.log(1.0 - y_pred_clipped))
        return loss

    def fit(self, X, y):
        n_samples, n_features = X.shape
        # Khởi tạo trọng số bằng 0
        self.weights = np.zeros(n_features)
        self.loss_history = []

        for i in range(self.n_iterations):
            # 1. Forward Pass: tính xác suất dự đoán
            linear_output = np.dot(X, self.weights)
            y_pred = self._sigmoid(linear_output)

            # 2. Tính toán giá trị hàm mất mát
            loss = self._compute_loss(y, y_pred)
            self.loss_history.append(loss)

            # 3. Tính Gradient theo công thức: (1/N) * X^T (y_pred - y)
            gradient = (1.0 / n_samples) * np.dot(X.T, (y_pred - y))

            # 4. Cập nhật trọng số theo Gradient Descent
            self.weights -= self.learning_rate * gradient

            # In tiến trình huấn luyện
            if self.verbose and (i % 100 == 0 or i == self.n_iterations - 1):
                print(f"Vòng lặp {i:4d}/{self.n_iterations} | Loss: {loss:.6f}")

        return self

    def predict_proba(self, X):
        linear_output = np.dot(X, self.weights)
        return self._sigmoid(linear_output)

    def predict(self, X, threshold=0.5):
        probabilities = self.predict_proba(X)
        return (probabilities >= threshold).astype(int)

## Bước 5: Huấn luyện mô hình và Trực quan hóa Hàm mất mát (Loss Curve)

In [ ]:
# Khởi tạo mô hình và huấn luyện
print("Bắt đầu huấn luyện mô hình Logistic Regression...")
model = LogisticRegressionNumPy(learning_rate=0.2, n_iterations=1000, verbose=True)
model.fit(X_train_b, y_train)

# Vẽ đường cong hàm mất mát (Loss Curve)
plt.figure(figsize=(10, 5))
plt.plot(model.loss_history, color='#1f77b4', linewidth=2.5, label='Training Loss')
plt.title('Đường cong Hàm Mất Mát (Loss Curve) theo số vòng lặp huấn luyện', fontsize=13, fontweight='bold')
plt.xlabel('Số vòng lặp (Iterations)', fontsize=11)
plt.ylabel('Binary Cross-Entropy Loss', fontsize=11)
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

## Bước 6: Đánh giá mô hình (Precision, Recall, F1-Score & Confusion Matrix)

Ta tự tính toán các chỉ số đánh giá phân loại nhị phân trên tập kiểm tra ($X_{test}, y_{test}$):
- **True Positive (TP)**: Thực tế có mưa và mô hình dự đoán có mưa.
- **False Positive (FP)**: Thực tế không mưa nhưng mô hình dự đoán có mưa (báo động nhầm).
- **True Negative (TN)**: Thực tế không mưa và mô hình dự đoán không mưa.
- **False Negative (FN)**: Thực tế có mưa nhưng mô hình dự đoán không mưa (bỏ sót).

Các công thức đánh giá:
- **Accuracy**: $\frac{TP + TN}{TP + TN + FP + FN}$
- **Precision**: $\frac{TP}{TP + FP}$ (Trong các lần báo mưa, tỷ lệ đúng là bao nhiêu?)
- **Recall**: $\frac{TP}{TP + FN}$ (Trong tất cả các lần trời mưa thực tế, bắt được bao nhiêu %?)
- **F1-Score**: $2 \times \frac{Precision \times Recall}{Precision + Recall}$

In [ ]:
# Dự đoán trên tập kiểm tra (Test Set)
y_pred_test = model.predict(X_test_b, threshold=0.5)

# Tính toán ma trận nhầm lẫn (Confusion Matrix)
TP = np.sum((y_test == 1) & (y_pred_test == 1))
FP = np.sum((y_test == 0) & (y_pred_test == 1))
TN = np.sum((y_test == 0) & (y_pred_test == 0))
FN = np.sum((y_test == 1) & (y_pred_test == 0))

# Tính toán các độ đo
accuracy = (TP + TN) / len(y_test)
precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

print("=" * 50)
print("KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH LOGISTIC REGRESSION (NUMPY)")
print("=" * 50)
print(f"- Accuracy  (Độ chính xác tổng) : {accuracy * 100:.2f}%")
print(f"- Precision (Độ chuẩn xác)       : {precision * 100:.2f}%")
print(f"- Recall    (Độ thu hồi / nhạy)  : {recall * 100:.2f}%")
print(f"- F1-Score  (Điểm F1)            : {f1:.4f}")
print("=" * 50)

# Trực quan hóa ma trận nhầm lẫn (Confusion Matrix)
cm = np.array([[TN, FP], [FN, TP]])
fig, ax = plt.subplots(figsize=(6, 5))
cax = ax.imshow(cm, cmap='Blues', interpolation='nearest')
fig.colorbar(cax)

for i in range(2):
    for j in range(2):
        val = cm[i, j]
        pct = val / len(y_test) * 100
        color = 'white' if val > cm.max() / 2 else 'black'
        ax.text(j, i, f"{val:,}\n({pct:.1f}%)", ha='center', va='center', 
                color=color, fontsize=12, fontweight='bold')

ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(['Không mưa (0)', 'Có mưa (1)'], fontsize=11)
ax.set_yticklabels(['Không mưa (0)', 'Có mưa (1)'], fontsize=11)
ax.set_xlabel('Dự đoán của mô hình (Predicted)', fontsize=12, fontweight='bold')
ax.set_ylabel('Thực tế quan trắc (Actual)', fontsize=12, fontweight='bold')
ax.set_title('Ma trận nhầm lẫn (Confusion Matrix)', fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

## Bước 7: Đối sánh với thư viện Scikit-Learn (Theo yêu cầu Assignment 3)

Chúng ta sử dụng `LogisticRegression` từ thư viện `sklearn.linear_model` để:
1. So sánh kết quả đánh giá (`accuracy`, `precision`, `recall`, `f1`).
2. So sánh hệ số trọng số ($w$) tìm được giữa mô hình tự code thuần NumPy và thuật toán của Scikit-Learn.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Huấn luyện mô hình của Scikit-Learn trên dữ liệu đã scale
sk_model = LogisticRegression(solver='lbfgs', max_iter=1000, random_state=42)
sk_model.fit(X_train_scaled, y_train)

# Dự đoán trên tập kiểm tra
y_pred_sk = sk_model.predict(X_test_scaled)

print("Báo cáo phân loại từ Scikit-Learn:")
print(classification_report(y_test, y_pred_sk, target_names=['Không mưa (0)', 'Có mưa (1)']))

# So sánh trọng số giữa NumPy scratch và Scikit-Learn
feature_names = ['Bias (Intercept)'] + feature_cols
print("\n" + "=" * 65)
print(f"{'Đặc trưng':<20} | {'Trọng số NumPy':<18} | {'Trọng số Sklearn':<18}")
print("=" * 65)
print(f"{feature_names[0]:<20} | {model.weights[0]:<18.4f} | {sk_model.intercept_[0]:<18.4f}")
for idx, fname in enumerate(feature_cols):
    print(f"{fname:<20} | {model.weights[idx+1]:<18.4f} | {sk_model.coef_[0][idx]:<18.4f}")
print("=" * 65)

## Bước 8: Phân tích Ý nghĩa Trọng số & Thử nghiệm Dự báo Tương tác

### Ý nghĩa các trọng số:
- **Độ ẩm (Humidity)**: Có trọng số dương rất lớn, phản ánh độ ẩm càng cao thì khả năng xuất hiện mưa càng lớn.
- **Áp suất khí quyển (Barometer)**: Có trọng số âm, phản ánh đặc điểm khí tượng học: khi áp suất khí quyển giảm (khí áp thấp) thường đi kèm mây đối lưu và mưa giông.
- **Tháng trong năm (Month)**: Phản ánh mùa mưa tại TP.HCM (Thủ Đức) tập trung vào các tháng giữa và cuối năm.

In [ ]:
def predict_rain_scenario(temperature, humidity, wind, barometer, hour, month):
    """
    Hàm nhận vào các chỉ số thời tiết thực tế và dự báo xác suất mưa bằng mô hình NumPy.
    """
    sample = np.array([[temperature, humidity, wind, barometer, hour, month]], dtype=float)
    # Chuẩn hoá theo mean và std của tập train
    sample_scaled = (sample - mean_train) / std_train
    # Bổ sung bias
    sample_b = np.c_[np.ones(1), sample_scaled]
    
    prob = model.predict_proba(sample_b)[0]
    decision = "🌧️ DỰ BÁO: CÓ MƯA" if prob >= 0.5 else "☀️ DỰ BÁO: KHÔNG MƯA"
    
    print(f"- Đầu vào: {temperature}°C, Độ ẩm {humidity}%, Gió {wind} km/h, Áp suất {barometer} hPa (Lúc {hour}h, Tháng {month})")
    print(f"  ==> Xác suất mưa: {prob * 100:.2f}% | {decision}\n")

print("--- THỬ NGHIỆM KỊCH BẢN THỜI TIẾT THỰC TẾ ---\n")
# Kịch bản 1: Mùa mưa, độ ẩm cao 94%, áp suất tụt xuống 1008 hPa, nhiệt độ dịu mát
predict_rain_scenario(temperature=26.0, humidity=94, wind=12.0, barometer=1008.0, hour=16, month=9)

# Kịch bản 2: Mùa khô, trưa nắng gắt 35°C, độ ẩm thấp 42%, áp suất cao 1013 hPa
predict_rain_scenario(temperature=35.0, humidity=42, wind=7.0, barometer=1013.5, hour=12, month=3)